In [ ]:
import requests

API_KEY = "s2k-QaPts7auX0ALWKflUBsme9YoIhIN6ZEc4mZMP77R"  # Replace with your actual key

# Test with a single paper request
url = "https://api.semanticscholar.org/graph/v1/paper/10.1038/nature12373"
headers = {"x-api-key": API_KEY}
params = {"fields": "title,abstract"}

response = requests.get(url, headers=headers, params=params)

print(f"Status Code: {response.status_code}")
print(f"Response: {response.text[:500]}")

if response.status_code == 200:
    print("✅ API key works!")
else:
    print("❌ API key failed - check your key")

In [ ]:
import requests
import time
import json
import os
from datetime import datetime
from typing import List, Dict, Optional
import re

# Your free API key
API_KEY = "s2k-QaPts7auX0ALWKflUBsme9YoIhIN6ZEc4mZMP77R"

URL = "https://api.semanticscholar.org/graph/v1/paper/batch"

def load_dois_from_file(filepath: str = "dois.txt") -> List[str]:
    """
    Load DOIs from a text file (one DOI per line)
    Skips empty lines and comments (lines starting with #)
    """
    dois = []
    
    if not os.path.exists(filepath):
        print(f"⚠️ File '{filepath}' not found!")
        return []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            # Skip empty lines and comments
            if line and not line.startswith('#'):
                dois.append(line)
    
    print(f"📂 Loaded {len(dois)} DOIs from {filepath}")
    return dois

def fetch_papers_in_batches(
    doi_list: List[str], 
    batch_size: int = 500, 
    delay: float = 0.5,
    max_retries: int = 3
) -> List[Dict]:
    """
    Fetch papers with API key for higher rate limits
    
    Args:
        doi_list: List of DOIs to fetch
        batch_size: Number of papers per batch (max 500)
        delay: Seconds between requests
        max_retries: Number of retry attempts for failed requests
    
    Returns:
        List of paper data dictionaries
    """
    if not doi_list:
        print("⚠️ No DOIs to fetch!")
        return []
    
    results = []
    total_batches = (len(doi_list) + batch_size - 1) // batch_size
    
    headers = {"x-api-key": API_KEY}
    
    for i in range(0, len(doi_list), batch_size):
        batch = doi_list[i:i+batch_size]
        payload = {"ids": batch}
        params = {"fields": "paperId,title,abstract,authors,year,venue,citationCount,externalIds,url,referenceCount"}
        
        retries = 0
        while retries < max_retries:
            try:
                print(f"⏳ Sending request {i//batch_size + 1}/{total_batches}...")
                response = requests.post(URL, json=payload, params=params, 
                                       headers=headers, timeout=30)
                
                if response.status_code == 200:
                    data = response.json()
                    # Handle both list and dict responses
                    if isinstance(data, list):
                        batch_results = data
                    else:
                        batch_results = data.get("data", [])
                    
                    # Filter out None values and ensure we have valid data
                    valid_results = [paper for paper in batch_results if paper is not None]
                    
                    # Log any None values found
                    if len(valid_results) < len(batch_results):
                        print(f"⚠️ Found {len(batch_results) - len(valid_results)} invalid entries (likely DOIs not found)")
                    
                    results.extend(valid_results)
                    print(f"✅ Batch {i//batch_size + 1}/{total_batches}: Fetched {len(valid_results)} papers")
                    break  # Success, exit retry loop
                    
                elif response.status_code == 429:
                    wait_time = delay * (2 ** retries)  # Exponential backoff
                    print(f"⏳ Rate limited! Waiting {wait_time:.1f} seconds...")
                    time.sleep(wait_time)
                    retries += 1
                    
                else:
                    print(f"❌ Batch {i//batch_size + 1}/{total_batches}: Error {response.status_code}")
                    print(f"   {response.text[:200]}")
                    break  # Don't retry on client errors (4xx)
                    
            except requests.exceptions.RequestException as e:
                print(f"❌ Batch {i//batch_size + 1}/{total_batches}: Request failed - {e}")
                retries += 1
                time.sleep(delay)
        
        # Rate limiting between batches
        if i + batch_size < len(doi_list):
            time.sleep(delay)
    
    return results

def generate_bibtex(paper: Dict, include_abstract: bool = True) -> str:
    """
    Generate BibTeX entry for a paper with format: Author.Year,
    
    Args:
        paper: Paper data dictionary
        include_abstract: Whether to include abstract in BibTeX entry
    
    Returns:
        BibTeX formatted string
    """
    # Get paper details with defaults
    title = paper.get('title', 'Untitled')
    authors = paper.get('authors', [])
    year = paper.get('year', 'n.d.')
    venue = paper.get('venue', '')
    doi = paper.get('externalIds', {}).get('DOI', '')
    url = paper.get('url', '')
    abstract = paper.get('abstract', '')
    
    # Format authors for BibTeX
    def format_author(author):
        name = author.get('name', '')
        if not name:
            return ''
        # Try to split into first and last names
        parts = name.split()
        if len(parts) >= 2:
            last = parts[-1]
            first = ' '.join(parts[:-1])
            return f"{last}, {first}"
        else:
            return name
    
    author_list = [format_author(a) for a in authors if a is not None]
    author_str = ' and '.join(author_list)
    
    # 🔥 Generate BibTeX key: Author.Year,
    # Get first author's last name
    if authors:
        first_author = authors[0].get('name', 'Unknown')
        # Extract last name
        if first_author:
            last_name = first_author.split()[-1] if ' ' in first_author else first_author
            # Clean last name (remove special characters)
            last_name = re.sub(r'[^a-zA-Z]', '', last_name)
        else:
            last_name = 'Unknown'
    else:
        last_name = 'Unknown'
    
    # Handle missing year
    year_str = str(year) if year and year != 'n.d.' else 'n.d.'
    
    # Create key: LastName.Year,
    key = f"{last_name}.{year_str}"
    
    # Build BibTeX entry
    bibtex = f"@article{{{key},\n"
    bibtex += f"  author = {{{author_str}}},\n"
    bibtex += f"  title = {{{title}}},\n"
    
    if year and year != 'n.d.':
        bibtex += f"  year = {{{year}}},\n"
    
    if venue:
        bibtex += f"  journal = {{{venue}}},\n"
    
    if doi:
        bibtex += f"  doi = {{{doi}}},\n"
    
    if url:
        bibtex += f"  url = {{{url}}},\n"
    
    if include_abstract and abstract:
        # Clean abstract for BibTeX (remove special characters that might break BibTeX)
        abstract_clean = abstract.replace('{', '\\{').replace('}', '\\}')
        abstract_clean = abstract_clean.replace('%', '\\%')
        bibtex += f"  abstract = {{{abstract_clean}}},\n"
    
    # Remove trailing comma and close
    bibtex = bibtex.rstrip(',\n') + '\n'
    bibtex += "}"
    
    return bibtex

def save_bibtex(papers: List[Dict], filename: str = "references_with_abstract.bib", include_abstract: bool = True):
    """
    Save papers as BibTeX entries
    
    Args:
        papers: List of paper data dictionaries
        filename: Output filename
        include_abstract: Whether to include abstracts in BibTeX
    """
    valid_papers = [p for p in papers if p is not None]
    
    if not valid_papers:
        print("No papers to export to BibTeX")
        return
    
    with open(filename, 'w', encoding='utf-8') as f:
        f.write("% BibTeX references generated from Semantic Scholar\n")
        f.write(f"% Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"% Total entries: {len(valid_papers)}\n\n")
        
        for paper in valid_papers:
            bibtex = generate_bibtex(paper, include_abstract)
            f.write(bibtex)
            f.write("\n\n")
    
    print(f"📚 BibTeX saved to {filename} ({len(valid_papers)} entries)")
    if include_abstract:
        print(f"   ✅ Abstracts included in BibTeX entries")
    else:
        print(f"   ⚠️ Abstracts excluded from BibTeX entries")

def save_results(papers: List[Dict], filename: str = None):
    """Save paper data to JSON file"""
    if not papers:
        print("⚠️ No papers to save!")
        return None
    
    if not filename:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"semantic_scholar_papers_{timestamp}.json"
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(papers, f, indent=2, ensure_ascii=False)
    
    print(f"💾 Saved {len(papers)} papers to {filename}")
    return filename

def print_paper_summary(papers: List[Dict], max_abstract_length: int = 300):
    """Print formatted paper summaries with BibTeX preview"""
    if not papers:
        print("No papers to display")
        return
    
    # Filter out None values
    valid_papers = [p for p in papers if p is not None]
    
    if not valid_papers:
        print("No valid papers to display")
        return
    
    print(f"\n📊 Total valid papers fetched: {len(valid_papers)}")
    if len(papers) != len(valid_papers):
        print(f"⚠️ Excluded {len(papers) - len(valid_papers)} invalid entries")
    
    print("="*80)
    
    for idx, paper in enumerate(valid_papers, 1):
        # Safe get with default values
        print(f"\n📄 Paper {idx}: {paper.get('title', 'Untitled')}")
        print(f"📌 DOI: {paper.get('externalIds', {}).get('DOI', 'N/A')}")
        print(f"📅 Year: {paper.get('year', 'N/A')}")
        print(f"📚 Venue: {paper.get('venue', 'N/A')}")
        print(f"📊 Citations: {paper.get('citationCount', 'N/A')}")
        
        authors = paper.get('authors', [])
        if authors:
            author_names = [a.get('name', '') for a in authors[:3] if a is not None]
            author_str = ", ".join(author_names)
            if len(authors) > 3:
                author_str += f" et al. ({len(authors)} authors)"
            print(f"✍️ Authors: {author_str}")
        
        abstract = paper.get('abstract', 'No abstract available')
        if abstract and abstract != 'No abstract available':
            if len(abstract) > max_abstract_length:
                abstract = abstract[:max_abstract_length] + "..."
            print(f"\n📝 Abstract: {abstract}")
        
        # Show BibTeX preview
        if idx <= 3:  # Show preview for first 3 papers
            print(f"\n📚 BibTeX Preview:")
            bibtex = generate_bibtex(paper, include_abstract=True)
            preview_lines = bibtex.split('\n')[:5]  # Show first 5 lines
            print(f"   {preview_lines[0]}")
            for line in preview_lines[1:3]:
                print(f"   {line}")
            print(f"   ...")
        
        print("-"*80)

def export_abstracts_to_text(papers: List[Dict], filename: str = "abstracts.txt"):
    """Export abstracts to a text file"""
    if not papers:
        print("No papers to export")
        return
    
    # Filter out None values
    valid_papers = [p for p in papers if p is not None]
    
    if not valid_papers:
        print("No valid papers to export")
        return
    
    with open(filename, 'w', encoding='utf-8') as f:
        for paper in valid_papers:
            title = paper.get('title', 'Untitled')
            abstract = paper.get('abstract', 'No abstract available')
            doi = paper.get('externalIds', {}).get('DOI', 'N/A')
            authors = paper.get('authors', [])
            author_names = [a.get('name', '') for a in authors if a is not None]
            author_str = ", ".join(author_names)
            
            f.write(f"Title: {title}\n")
            f.write(f"Authors: {author_str}\n")
            f.write(f"DOI: {doi}\n")
            f.write(f"Abstract: {abstract}\n")
            f.write("="*80 + "\n\n")
    
    print(f"📝 Abstracts exported to {filename} (total: {len(valid_papers)})")

def identify_missing_dois(original_dois: List[str], fetched_papers: List[Dict]) -> List[str]:
    """Identify which DOIs were not found"""
    found_dois = set()
    for paper in fetched_papers:
        if paper is not None:
            doi = paper.get('externalIds', {}).get('DOI')
            if doi:
                found_dois.add(doi)
    
    missing_dois = [doi for doi in original_dois if doi not in found_dois]
    return missing_dois

def main():
    """Main execution function"""
    print("="*80)
    print("📚 Semantic Scholar Paper Fetcher with BibTeX Export")
    print("="*80)
    
    # Load DOIs from file
    dois = load_dois_from_file("dois.txt")
    
    if not dois:
        print("❌ No DOIs found in dois.txt")
        print("   Please create dois.txt with one DOI per line")
        return
    
    print(f"📋 Total DOIs to process: {len(dois)}\n")
    
    # Ask user if they want abstracts in BibTeX
    include_abstract = True  # Default to True
    
    # Fetch papers
    start_time = time.time()
    papers = fetch_papers_in_batches(dois, delay=0.5)
    elapsed = time.time() - start_time
    
    print(f"\n⏱️ Completed in {elapsed:.2f} seconds")
    
    # Save results
    if papers:
        # Filter out None values before saving
        valid_papers = [p for p in papers if p is not None]
        
        # Save to JSON
        json_file = save_results(valid_papers)
        
        # Print summaries
        print_paper_summary(valid_papers)
        
        # Export abstracts to text file
        export_abstracts_to_text(valid_papers)
        
        # 🔥 Export to BibTeX with format: Author.Year,
        save_bibtex(valid_papers, "references_with_abstract.bib", include_abstract=include_abstract)
        
        # Also create a version without abstracts (for submission)
        save_bibtex(valid_papers, "references_no_abstract.bib", include_abstract=False)
        
        # Identify missing DOIs
        missing_dois = identify_missing_dois(dois, valid_papers)
        if missing_dois:
            print(f"\n⚠️ {len(missing_dois)} DOIs were not found:")
            print(f"   {missing_dois[:10]}")  # Show first 10
            if len(missing_dois) > 10:
                print(f"   ... and {len(missing_dois) - 10} more")
            
            # Save missing DOIs to file
            with open("missing_dois.txt", "w", encoding='utf-8') as f:
                for doi in missing_dois:
                    f.write(doi + "\n")
            print(f"💾 Missing DOIs saved to missing_dois.txt")
        
        # Show summary statistics
        print("\n📊 Summary Statistics:")
        print(f"   Total DOIs requested: {len(dois)}")
        print(f"   Papers successfully fetched: {len(valid_papers)}")
        print(f"   Papers with abstracts: {sum(1 for p in valid_papers if p.get('abstract'))}")
        print(f"   Papers without abstracts: {sum(1 for p in valid_papers if not p.get('abstract'))}")
        print(f"   BibTeX entries generated: 2 files (with/without abstracts)")
        print(f"   BibTeX key format: Author.Year,")
        
        # Show count by year
        years = {}
        for p in valid_papers:
            year = p.get('year')
            if year:
                years[year] = years.get(year, 0) + 1
        if years:
            print("\n   Papers by year:")
            for year in sorted(years.keys(), reverse=True)[:10]:  # Show top 10 years
                print(f"      {year}: {years[year]} papers")
        
    else:
        print("❌ No papers fetched. Please check:")
        print("   1. Your API key is valid")
        print("   2. The DOIs in dois.txt are correct")
        print("   3. You have internet connection")

# Run the main function
if __name__ == "__main__":
    main()